# PR07 · Measure volume before making an atrophy claim

<!-- paper-first -->
### Begin with the paper question

**Read or revisit [PP03](../../curriculum/papers/processing.md#pp03).** Use the assigned first-pass sections in the guide; if you already read them, return only to the relevant figure or claim. Do this before the technical explanation below.

**Motivation:** What extra measurement assumptions are needed to turn a segmentation into a volume comparison?

Write a two-sentence prediction and one thing you cannot yet explain. Ask your AI tutor to locate evidence in the supplied paper and distinguish it from inference. A paper link motivates this question; it does not mean the paper uses every method demonstrated here.

**After the experiment:** revisit your prediction in the [evidence ledger](../../curriculum/coursework/EVIDENCE_LEDGER.md). Explain one mechanism you now understand, cite a result from this notebook, and name a paper claim this exercise still cannot test. Keep a small demonstration distinct from a reproduction of the study.
<!-- /paper-first -->

**Format:** 75–100 minutes for this lesson and its small executable lab, followed by the explicitly labeled upstream practice. Run this notebook from a fresh kernel, top to bottom. Core data are synthetic. Software: NumPy, SciPy, Matplotlib; extra imports are stated in code. This is one stage of a longer processing course, not a replacement for supervised research training.

## Learning objectives

- Convert fractional occupancy to physical volume.
- Explain modulation using the direction of a coordinate mapping.
- Identify why cross-sectional and longitudinal morphology require different controls.

## Understand the operation

Morphometry asks how anatomical measurements differ, but the measurement has to be defined first. A hard-label voxel count, a fractional gray-matter volume, cortical thickness and a deformation-derived local volume factor are different quantities. None can be silently substituted for another. Physical voxel volume comes from spatial geometry; the same count on a different grid can represent a different amount of tissue.

Warping changes coordinate volume. In a simple forward map with Jacobian determinant J, a material density must scale by1/J to preserve integrated mass when the corresponding volume element scales byJ. Tool descriptions may use an inverse map, so blindly multiplying by a file called “Jacobian” can reverse the intended correction. Voxel-based morphometry pipelines define precisely which field, affine contribution and modulation convention they use. The lab demonstrates conservation for corresponding material cells, not a complete resampled VBM image.

For longitudinal change, processing two time points independently can introduce asymmetric registration and segmentation effects. A subject-specific unbiased template and consistent processing may reduce some biases, but neither ensures changes are biological. Scanner changes, head position, motion, hydration, lesion evolution and model failures can alter measured morphology. Baseline intracranial size and other covariates require a study-specific statistical plan rather than automatic division. Inspect the individual measurement before aggregating across people, and keep the distinction between measurement uncertainty and group inference visible.

## Transformation contract

**Input:** fractional occupancy values and a spatial affine. **Output:** volume in mm³, plus an analytic modulation demonstration. **Preserved under the correct coordinate change:** integrated tissue amount. **Lost by hard thresholding:** fractional boundary contributions. **Not performed:** image normalization, statistical VBM or longitudinal FreeSurfer.


## Read the actual course material

- [Oxford FSL: Structural Analysis practical](https://pages.fmrib.ox.ac.uk/fslcourse/practicals/seg_struc/index.html). Public university practical; linked only.
- [FreeSurfer: volume/surface, ROI and longitudinal workshop](https://surfer.nmr.mgh.harvard.edu/fswiki/FsTutorial). Official workshop; linked only.

Read the named topic alongside this lesson; compare its real-image assumptions with our controlled example. These notebooks use original explanations and original code, not copied upstream passages. The source chapter is the place to continue to a complete real-tool practical. External software and downloaded datasets are not silently run by this notebook.


## Predict, then ask your AI assistant

Use Goose with your installed Ollama model, or ChatGPT. The model is a tutor and code author; the local Python runtime performs these calculations. Paste:

> Compute fractional tissue volume using affine-derived voxel volume. Derive the correct density factor for the stated forward Jacobian and demonstrate mass conservation. Keep this analytic material-cell example distinct from resampling a real VBM image. Return at most 20 executable lines per cell, show units and array shapes, and preserve the original. Explain the prediction before running. If an assertion fails, diagnose the disagreement rather than deleting the check.

Write your prediction before executing the reference cells below.


In [1]:
import numpy as np
pve=np.array([.1,.3,.6,.9,1.0])
A=np.diag([2.,2.,3.,1.])
voxel_volume=abs(np.linalg.det(A[:3,:3]))
fractional_volume=pve.sum()*voxel_volume
threshold_volume=np.count_nonzero(pve>.5)*voxel_volume
print('fractional / threshold volume mm^3:',fractional_volume,threshold_volume)
assert np.isclose(fractional_volume,34.8) and np.isclose(threshold_volume,36)
J=np.array([.8,1.,1.2,1.5,2.])
new_cell_volume=voxel_volume*J
modulated_density=pve/J
mass=np.sum(modulated_density*new_cell_volume)
unmodulated=np.sum(pve*new_cell_volume)
assert np.isclose(mass,fractional_volume) and not np.isclose(unmodulated,mass)
print('conserved vs unmodulated amount:',mass,unmodulated)


fractional / threshold volume mm^3: 34.8 36.0
conserved vs unmodulated amount: 34.8 53.4


In [2]:
baseline=np.array([1000.,1100.,900.])
followup_true=baseline*.98
scanner_scale=1.03
observed_followup=followup_true*scanner_scale
true_change=100*(followup_true-baseline)/baseline
observed_change=100*(observed_followup-baseline)/baseline
print('true / observed percentage changes:',true_change,observed_change)
assert np.all(true_change<0) and np.all(observed_change>0)
print('A scanner-dependent measurement effect can reverse the apparent direction.')


true / observed percentage changes: [-2. -2. -2.] [0.94 0.94 0.94]
A scanner-dependent measurement effect can reverse the apparent direction.


## Check and explain

The fractional volume is34.8mm³ while thresholding produces36mm³. Modulated density times transformed cell volume recovers34.8. The constructed scanner scale reverses the sign of a true2% loss; it is a demonstration of measurement confounding, not an estimated scanner correction.

## Deliberately wrong method

Reporting voxel count as mm³ or applying modulation without its mapping direction is wrong. The time-point example shows why a percentage difference is not automatically atrophy. You cannot infer and subtract a scanner effect from one pair merely because the curve looks plausible.

## Transfer to an actual dataset or tool — guided assignment

Follow Oxford FSL-VBM’s provided structural outputs and identify segmentation, template creation, nonlinear registration, modulation and smoothing stages. For longitudinal interest, read the FreeSurfer longitudinal tutorial linked from the workshop and write a within-subject processing plan. Inspect one subject’s maps and record whether affine scaling is included in the chosen morphometry metric. Full VBM/longitudinal pipelines and group inference remain external practicals.

**Submit:** a transformation card, one labeled figure or numerical result, the failed-method diagnosis, and the upstream-practice evidence. If the external exercise has not been run, mark it **not executed** and state the missing software/data; do not convert a proposed command into a claimed result.

## Exit questions and answer key

1. Why does .6 occupancy contribute less than one voxel? **Answer:** it represents fractional tissue under the stated PVE model.
2. Why must the Jacobian direction be known? **Answer:** forward and inverse volume factors are reciprocals at corresponding positions; using the wrong one violates the intended conservation.


### Return to the research question

Reopen [PP03](../../curriculum/papers/processing.md#pp03) and your initial two-sentence prediction. In your [evidence ledger](../../curriculum/coursework/EVIDENCE_LEDGER.md):

1. Cite one output or diagnostic from this lesson and explain the transformation it demonstrates.
2. Revise one claim or question from the paper, with a figure/section locator. State what this small exercise still cannot establish about the published result.
3. Ask AI to propose a next check. Accept, revise or reject it with a scientific reason. Then explain your decision aloud without reading the AI response.

Reuse this entry in the A2 portfolio when relevant; a separate report is unnecessary.
